# Water Rights Restored to the Gila River

The impacts of irrigation on vegetation health in the Gila River Valley

## Set up

To get started on this notebook, you’ll need to restore any variables from previous notebooks to your workspace. To save time and memory, make sure to specify which variables you want to load.

In [1]:
%store -r boundary_gdf project

You will also need to import any libraries you are using in this
notebook, since they won’t carry over from the previous notebook:

In [2]:
### import libraries
### working with dataframes
import pandas as pd

### working with geodataframes
import geopandas as gpd

### making interactive plots of tabular and vector data
import hvplot.pandas

### work with json data
import json

### work with pattern matching rules for pathnames 
from glob import glob

### access the data we'll use for the assignment
import earthpy

### interactive plots with rasters 
import hvplot.xarray

### handle geospatial rasters
import rioxarray as rxr

### work with arrays
import xarray as xr

## Step 3: Wrangle the raster data

### Step 3A: Load NDVI data

When you ran the code `project.get_data()` in the previous notebook, you actually downloaded the NDVI data in addition to the Gila River Indian Community boundary data. Now you need to load all these downloaded NDVI files into Python. Let’s start by getting all the file names. You will also need to extract the date from the filename. Check out [the lesson on getting information from filenames from EarthDataScience.org](https://www.earthdatascience.org/courses/intro-to-earth-data-science/write-efficient-python-code/loops/data-workflows-with-loops/).

In this lesson, you will use the `glob.glob()` function in Python to find all files that match a **pattern** formed with the **wildcard** character `*`. The wildcard can represent any string of alphanumberic characters. For example, the pattern `'file_*.tif'` will match the files `'file_1.tif'`, `'file_2.tif'`, or even `'file_qeoiurghtfoqaegbn34pf.tif'`, but it will not match `'something-else.csv'` or even `'something-else.tif'`.

In this notebook, we’ll use the `.rglob()`, or **recursive** glob method of the Path object instead. It works similarly, but you’ll notice that we have to convert the results to a list with the `list()` function.

**Watch out!**: `glob` doesn’t necessarily find files in the order you would expect. Make sure to **sort** your file names.

### Reflect and Respond #5:
Take a look at the file names for the NDVI files. If you need a refresher on where earthpy downloaded the files to, you can make a new code chunk and run `print(project.project_dir)` again. What do you notice is the same for all the files? Keep in mind that you only want the NDVI files, not the Quality files (which will flag potential incorrect measurements).

All the files start with "MOD13Q1.061__250m_16_days_" and end with "_aid0001.tif. In between are dates and also there seem to be two categories - those with NDVI in the name and those with "VI_quality" in the name.

- Create a **pattern** for the files you want to import. Your pattern should include the parts of the file names that are the same for all files, and replace the rest with the `*` character. Make sure to match the NDVI files, but not the Quality files!  
- Replace `ndvi-pattern` with your pattern  
- Run the code and make sure that you are getting all the files you want and none of the files you don’t

In [3]:
### get a sorted list of NDVI tif file paths
ndvi_paths = sorted(list(project.project_dir.rglob('*NDVI*.tif')))

### display the first and last three files paths to check the pattern
ndvi_paths[:3], ndvi_paths[-3:]

([WindowsPath('C:/Users/honey/AppData/Local/earth-analytics/earth-analytics/gila_data/gila-river-ndvi/MOD13Q1.061_2001137_to_2022244/MOD13Q1.061__250m_16_days_NDVI_doy2001145000000_aid0001.tif'),
  WindowsPath('C:/Users/honey/AppData/Local/earth-analytics/earth-analytics/gila_data/gila-river-ndvi/MOD13Q1.061_2001137_to_2022244/MOD13Q1.061__250m_16_days_NDVI_doy2001161000000_aid0001.tif'),
  WindowsPath('C:/Users/honey/AppData/Local/earth-analytics/earth-analytics/gila_data/gila-river-ndvi/MOD13Q1.061_2001137_to_2022244/MOD13Q1.061__250m_16_days_NDVI_doy2001177000000_aid0001.tif')],
 [WindowsPath('C:/Users/honey/AppData/Local/earth-analytics/earth-analytics/gila_data/gila-river-ndvi/MOD13Q1.061_2001137_to_2022244/MOD13Q1.061__250m_16_days_NDVI_doy2022209000000_aid0001.tif'),
  WindowsPath('C:/Users/honey/AppData/Local/earth-analytics/earth-analytics/gila_data/gila-river-ndvi/MOD13Q1.061_2001137_to_2022244/MOD13Q1.061__250m_16_days_NDVI_doy2022225000000_aid0001.tif'),
  WindowsPath('C:/U

In [4]:
# ### id files with name pattern
# ndvi_files = project.project_dir.rglob('*NDVI*.tif')
# # ndvi_files

# ### put matching files paths in list
# ndvi_list = list(ndvi_files)
# ndvi_list

# ### make sorted list of files
# ndvi_sorted = sorted(ndvi_list)
# ndvi_sorted

### Step 3B: Iterate over all the files to prep them for the analysis 
Now you should have a few dozen files! For each file, you need to:

-   Load the file in using the `rioxarray` library
-   Get the date from the file name
-   Add the date as a dimension coordinate
-   Give your data variable a name

You don’t want to write out the code for each file! That’s a recipe for copy-and-paste issues. Luckily, Python has tools for doing similar tasks repeatedly. In this case, you’ll use one called a `for` loop.

There’s some code below that uses a `for` loop in what is called an **accumulation pattern** to process each file. That means that you will save the results of your processing to a list each time you process the files, and then merge all the arrays in the list.


- Look at the file names. How many characters from the end is the date? `doy_start` and `doy_end` are used to extract the day of the year (doy) from the file name. You will need to count characters from the end and change the values to get the right part of the file name. HINT: the index -1 in Python means the last value, -2 second-to-last, and so on.  
- Replace any required variable names with your chosen variable names

In [5]:
### MOD13Q1.061_2001137_to_2022244/MOD13Q1.061__250m_16_days_NDVI_doy2001145000000_aid0001.tif
### counting from the end of the file name with the last index being -1 we can determine where the day of year starts and ends

In [6]:
### set the start and end day of year 
### based on the location of the dates in the filename pattern
doy_start = -25
doy_end = -19

### make empty list to fill with for loop
ndvi_das = []

### loop through each NDVI image
for ndvi_path in ndvi_paths:
    
    ### get date from file name:
    doy = ndvi_path.name[doy_start:doy_end]

    ### convert to date:
    date = pd.to_datetime(doy, format = '%Y%j')

    ### open dataset:
    da = rxr.open_rasterio(ndvi_path, mask_and_scale= True).squeeze() ### removes label of raster band layer - removing layer with dimension of 1

    ### add date dimension:
    da = da.assign_coords({'date': date})

    ### clean up metadata:
    da = da.expand_dims({'date': 1}) 

    ### name data variable
    da.name = 'NDVI'

    ### prep to concatenate the processed file to the list of files:
    ndvi_das.append(da)

### how big is ndvi_das
len(ndvi_das)

154

In [7]:
ndvi_das

[<xarray.DataArray 'NDVI' (date: 1, y: 203, x: 382)> Size: 310kB
 array([[[0.8282    , 0.6146    , 0.3796    , ..., 0.1542    ,
          0.1542    , 0.1774    ],
         [0.52349997, 0.32029998, 0.46359998, ..., 0.145     ,
          0.145     , 0.145     ],
         [0.52349997, 0.38509998, 0.4567    , ..., 0.0962    ,
          0.058     , 0.0641    ],
         ...,
         [0.13759999, 0.13759999, 0.1283    , ..., 0.1489    ,
          0.1489    , 0.1568    ],
         [0.1299    , 0.1299    , 0.12379999, ..., 0.1831    ,
          0.1831    , 0.23169999],
         [0.1347    , 0.1309    , 0.1329    , ..., 0.15709999,
          0.2384    , 0.2452    ]]], shape=(1, 203, 382), dtype=float32)
 Coordinates:
   * date         (date) datetime64[us] 8B 2001-01-14
   * y            (y) float64 2kB 33.39 33.39 33.38 33.38 ... 32.97 32.97 32.97
   * x            (x) float64 3kB -112.3 -112.3 -112.3 ... -111.5 -111.5 -111.5
     band         int64 8B 1
     spatial_ref  int64 8B 0
 Attribut

### Step 3C: Combine rasters

Next, stack your arrays by date into a time series using the `xr.combine_by_coords()` function. You will have to tell it which dimension you want to stack your data in.

In [8]:
### combine NDVI images from all dates
ndvi_da =xr.combine_by_coords(ndvi_das, coords= ['date'])

### check it out
ndvi_da

C:\Users\honey\AppData\Local\Temp\ipykernel_30100\3973762722.py:2: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ndvi_da =xr.combine_by_coords(ndvi_das, coords= ['date'])
C:\Users\honey\AppData\Local\Temp\ipykernel_30100\3973762722.py:2: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ndvi_da =xr.combine_by_coords(ndvi_das, coords= ['da

<xarray.Dataset> Size: 48MB
Dimensions:      (date: 154, y: 203, x: 382)
Coordinates:
  * date         (date) datetime64[us] 1kB 2001-01-14 2001-01-16 ... 2022-01-24
  * y            (y) float64 2kB 33.39 33.39 33.38 33.38 ... 32.97 32.97 32.97
  * x            (x) float64 3kB -112.3 -112.3 -112.3 ... -111.5 -111.5 -111.5
    band         int64 8B 1
    spatial_ref  int64 8B 0
Data variables:
    NDVI         (date, y, x) float32 48MB 0.8282 0.6146 ... 0.2146 0.2085

## Wrap up Step 3
Don’t forget to store your variables so you can use them in other notebooks! Replace `var1` and `var2` with the variable you want to save, separated by spaces.

In [9]:
%store ndvi_da

Stored 'ndvi_da' (Dataset)


Finally, be sure to `Restart` and `Run all` to make sure your notebook
works all the way through!